In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version

'3.5.0'

In [2]:
import os

API_BASE_URL = os.environ.get("API_BASE_URL", "http://mock-api:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

In [5]:
import requests

# Authenticate
resp = requests.post(f"{API_BASE_URL}/api/v1/auth/token", json={
    "username": API_USERNAME,
    "password": API_PASSWORD
})
token = resp.json()["access_token"]

# Fetch first page of orders
headers = {"Authorization": f"Bearer {token}"}
data = requests.get(f"{API_BASE_URL}/api/v1/data/orders", headers=headers, params={"page": 1, "page_size": 1000})
print(data.json()["pagination"])

{'page': 1, 'page_size': 1000, 'total_records': 99441, 'total_pages': 100, 'has_next': True}


In [6]:
import pandas as pd
from datetime import datetime
import os

BRONZE_PATH = "output/bronze"
MANIFEST_PATH = f"{BRONZE_PATH}/manifest.json"

MEDALLION_ARCHITECTURE= ["bronze","silver","gold"]

OUTPUT_PATH = "output"

for type in MEDALLION_ARCHITECTURE:
    if not os.path.exists(f"{OUTPUT_PATH}/{type}"):
        os.mkdir(f"{OUTPUT_PATH}/{type}")

def save_bronze(data, endpoint):
    df = pd.DataFrame(data)
    df["_ingested_at"] = datetime.utcnow()
    df["_source_endpoint"] = endpoint

    path = f"{BRONZE_PATH}/{endpoint}.csv"
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

In [7]:
import requests
import time
import random
import json
import pandas as pd
from datetime import datetime, timedelta

In [8]:
class APIClient:
    def __init__(self):
        self.base_url = API_BASE_URL
        self.username = API_USERNAME
        self.password = API_PASSWORD
        self.token = None
        self.get_token()

    def get_token(self):
        resp = requests.post(
            f"{self.base_url}/api/v1/auth/token",
            json={"username": self.username, "password": self.password}
        )
        self.token = resp.json()["access_token"]

    def headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def request(self, endpoint, params):
        for attempt in range(5):
            r = requests.get(
                f"{self.base_url}/api/v1/data/{endpoint}",
                headers=self.headers(),
                params=params
            )

            if r.status_code == 200:
                return r.json()

            elif r.status_code == 401:
                self.get_token()

            elif r.status_code in [500, 429]:
                sleep_time = (2 ** attempt) + random.uniform(0, 1)
                time.sleep(sleep_time)

            else:
                raise Exception(f"Error: {r.text}")

        raise Exception("Max retries exceeded")

In [9]:
def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

DEFAULT_START_DATE = "2018-07-01"

def get_last_date(manifest, endpoint):
    return manifest.get(endpoint, {}).get("last_date", DEFAULT_START_DATE)

def update_manifest(manifest, endpoint, new_date):
    manifest.setdefault(endpoint, {})["last_date"] = new_date

In [10]:
def fetch_data(client, endpoint, date_from=None):
    page = 1
    all_data = []

    while True:
        params = {
            "page": page,
            "page_size": 1000
        }

        if endpoint in ["orders", "order_items"]:
            params["date_from"] = date_from

        res = client.request(endpoint, params)
        data = res.get("data", [])

        if not data:
            break

        all_data.extend(data)

        if page >= res["pagination"]["total_pages"]:
            break

        page += 1

    return all_data

In [11]:
def remove_existing_records(df_new, endpoint, key_cols):
    path = f"{BRONZE_PATH}/{endpoint}.csv"

    if not os.path.exists(path):
        return df_new

    df_existing = pd.read_csv(path, usecols=key_cols)

    for col in key_cols:
        df_new[col] = df_new[col].astype(str)
        df_existing[col] = df_existing[col].astype(str)

    merged = df_new.merge(
        df_existing,
        on=key_cols,
        how="left",
        indicator=True
    )

    return merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])

In [12]:
def save_bronze(df, endpoint):
    if df.empty:
        return

    df["_ingested_at"] = datetime.utcnow()
    df["_source_endpoint"] = endpoint

    path = f"{BRONZE_PATH}/{endpoint}.csv"

    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

In [13]:
def extract_latest_date(df, endpoint):
    if df.empty:
        return None

    if endpoint == "orders":
        return df["order_purchase_timestamp"].max()[:10]

    if endpoint == "order_items":
        return df["shipping_limit_date"].max()[:10]

    return None

In [14]:
def adjust_date(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return (dt - timedelta(days=1)).strftime("%Y-%m-%d")

In [15]:
client = APIClient()
manifest = load_manifest()

endpoints = {
    "order_items": ["order_id", "order_item_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "payments": ["order_id", "payment_sequential"],
    "orders": ["order_id"]
}

for endpoint, keys in endpoints.items():

    print(f"\nProcessing: {endpoint}")

    last_date = get_last_date(manifest, endpoint)

    last_date = adjust_date(last_date)

    print(f"Fetching from: {last_date}")

    data = fetch_data(client, endpoint, last_date)

    if not data:
        print("No new data")
        continue

    df = pd.DataFrame(data)

    df = remove_existing_records(df, endpoint, keys)

    if df.empty:
        print("No new unique records")
        continue

    save_bronze(df, endpoint)

    new_date = extract_latest_date(df, endpoint)

    if new_date:
        update_manifest(manifest, endpoint, new_date)

    print(f"Inserted {len(df)} new rows")

save_manifest(manifest)


Processing: order_items
Fetching from: 2018-06-30
Inserted 15590 new rows

Processing: customers
Fetching from: 2018-06-30
Inserted 99441 new rows

Processing: products
Fetching from: 2018-06-30
Inserted 32951 new rows

Processing: sellers
Fetching from: 2018-06-30
Inserted 3095 new rows

Processing: payments
Fetching from: 2018-06-30
Inserted 103886 new rows

Processing: orders
Fetching from: 2018-06-30
Inserted 12948 new rows
